In [1]:
# ==========================================
# Install Required Libraries
# ==========================================

!pip install -q transformers==4.41.2
!pip install -q datasets==2.20.0
!pip install -q sentencepiece
!pip install -q scikit-learn
!pip install -q matplotlib
!pip install -q tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 25.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.5.0 which is incompatible.


In [2]:
# ==========================================
# Import Libraries
# ==========================================

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    AdamW
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [3]:
# ==========================================
# Check GPU
# ==========================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Device: cuda
Tesla T4


In [4]:
# ==========================================
# Mount Google Drive
# ==========================================

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# ==========================================
# Load Dataset
# ==========================================

train_df = pd.read_csv(
    "/content/drive/MyDrive/Fake-News-RAG/datasets/train_clean.csv"
)

valid_df = pd.read_csv(
    "/content/drive/MyDrive/Fake-News-RAG/datasets/valid_clean.csv"
)

test_df = pd.read_csv(
    "/content/drive/MyDrive/Fake-News-RAG/datasets/test_clean.csv"
)

print(train_df.head())

print()

print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)

                                                text  label
0  Says the Annies List political group supports ...      1
1  When did the decline of coal start? It started...      0
2  Hillary Clinton agrees with John McCain "by vo...      0
3  Health care reform legislation is likely to ma...      1
4  The economic turnaround started at the end of ...      0

(10240, 2)
(1284, 2)
(1267, 2)


In [6]:
# ==========================================
# Reproducibility
# ==========================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [7]:
# ==========================================
# Load BERT Tokenizer
# ==========================================

from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

print("✅ Tokenizer loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

✅ Tokenizer loaded successfully!


In [8]:
# ==========================================
# Custom Dataset Class
# ==========================================

from torch.utils.data import Dataset

class FakeNewsDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_length=128):

        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [9]:
# ==========================================
# Create Dataset Objects
# ==========================================

train_dataset = FakeNewsDataset(
    train_df,
    tokenizer
)

valid_dataset = FakeNewsDataset(
    valid_df,
    tokenizer
)

test_dataset = FakeNewsDataset(
    test_df,
    tokenizer
)

print(len(train_dataset))
print(len(valid_dataset))
print(len(test_dataset))

10240
1284
1267


In [10]:
# ==========================================
# DataLoaders
# ==========================================

from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE
)

print("✅ DataLoaders ready!")

✅ DataLoaders ready!


In [11]:
# ==========================================
# Load BERT Model
# ==========================================

from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model = model.to(device)

print("✅ Model loaded successfully!")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded successfully!


In [12]:
# ==========================================
# Optimizer
# ==========================================

from transformers import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

EPOCHS = 3

print("✅ Optimizer ready!")

/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


✅ Optimizer ready!


In [13]:
# ==========================================
# Training Function
# ==========================================

def train_epoch(model, dataloader, optimizer):

    model.train()

    total_loss = 0

    for batch in tqdm(dataloader):

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        total_loss += loss.item()

        loss.backward()

        optimizer.step()

    avg_loss = total_loss / len(dataloader)

    return avg_loss

In [14]:
# ==========================================
# Validation Function
# ==========================================

def evaluate(model, dataloader):

    model.eval()

    predictions = []
    true_labels = []

    total_loss = 0

    with torch.no_grad():

        for batch in tqdm(dataloader):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

            total_loss += loss.item()

            logits = outputs.logits

            preds = torch.argmax(
                logits,
                dim=1
            )

            predictions.extend(
                preds.cpu().numpy()
            )

            true_labels.extend(
                labels.cpu().numpy()
            )

    avg_loss = total_loss / len(dataloader)

    accuracy = accuracy_score(
        true_labels,
        predictions
    )

    precision = precision_score(
        true_labels,
        predictions
    )

    recall = recall_score(
        true_labels,
        predictions
    )

    f1 = f1_score(
        true_labels,
        predictions
    )

    return (
        avg_loss,
        accuracy,
        precision,
        recall,
        f1
    )

In [ ]:
# ==========================================
# Training Loop
# ==========================================

best_f1 = 0

for epoch in range(EPOCHS):

    print("=" * 50)
    print(f"Epoch {epoch + 1}/{EPOCHS}")

    train_loss = train_epoch(
        model,
        train_loader,
        optimizer
    )

    val_loss, acc, prec, rec, f1 = evaluate(
        model,
        valid_loader
    )

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Accuracy   : {acc:.4f}")
    print(f"Precision  : {prec:.4f}")
    print(f"Recall     : {rec:.4f}")
    print(f"F1 Score   : {f1:.4f}")

    if f1 > best_f1:

        best_f1 = f1

        save_path = "/content/drive/MyDrive/Fake-News-RAG/models"

        os.makedirs(
            save_path,
            exist_ok=True
        )

        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)

        print("✅ Best model saved!")

Epoch 1/3


100%|██████████| 81/81 [00:10<00:00,  7.71it/s]


Train Loss : 0.6814
Val Loss   : 0.6601
Accuracy   : 0.6106
Precision  : 0.5736
Recall     : 0.7338
F1 Score   : 0.6439
✅ Best model saved!
Epoch 2/3


100%|██████████| 81/81 [00:10<00:00,  7.68it/s]


Train Loss : 0.6463
Val Loss   : 0.6337
Accuracy   : 0.6433
Precision  : 0.6479
Recall     : 0.5617
F1 Score   : 0.6017
Epoch 3/3


 80%|████████  | 512/640 [03:01<00:45,  2.83it/s]

In [16]:
# ==========================================
# Test Evaluation
# ==========================================

test_loss, acc, prec, rec, f1 = evaluate(
    model,
    test_loader
)

print()

print("===== Test Results =====")

print(f"Loss      : {test_loss:.4f}")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1 Score  : {f1:.4f}")

100%|██████████| 80/80 [00:09<00:00,  8.33it/s]



===== Test Results =====
Loss      : 0.6409
Accuracy  : 0.6496
Precision : 0.6147
Recall    : 0.5280
F1 Score  : 0.5681


In [17]:
# ==========================================
# Inference Example
# ==========================================

text = "The government announced a new policy today."

encoding = tokenizer(
    text,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)

model.eval()

with torch.no_grad():

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

prediction = torch.argmax(
    outputs.logits,
    dim=1
).item()

label = "Fake News" if prediction == 1 else "Real News"

print("Prediction:", label)

Prediction: Fake News
